# 📡 Project 8.4 — Signal Pattern Analyser
**DSA for Mechatronics · Week 08 — Hash Tables & Dictionaries**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
from collections import Counter, defaultdict, OrderedDict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A signal processing pipeline analyses digitised sensor streams for fault patterns.
Four hash-map-powered algorithms are applied:

1. **Subarray sum equals K** — find all contiguous sub-windows whose amplitude sum equals a target (prefix-sum + hash map, O(n))
2. **Rolling hash window** — slide a fixed-width window over the stream and detect repeating patterns in O(n)
3. **Longest consecutive sequence** — find the longest run of consecutive sample IDs present in the log (hash set, O(n))
4. **Word-pattern bijection** — check if a fault-code sequence follows the same repeat pattern as a given template (two-way hash map)


## Step 1 — Subarray sum equals K (prefix sum + hash map)

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
N_SAMPLES    = random.randint(30, 50)
AMPLITUDE    = random.randint(5, 15)
TARGET_K     = random.randint(8, 20)
WINDOW_WIDTH = random.randint(4, 8)
N_CODES      = random.randint(20, 35)
CODE_POOL    = [f"E{i:02d}" for i in range(1, 12)]   # fault codes

# Signal: integers in [-AMPLITUDE, AMPLITUDE]
signal = [random.randint(-AMPLITUDE, AMPLITUDE) for _ in range(N_SAMPLES)]

def subarray_sum_k(arr, k):
    """
    Count subarrays with sum == k using prefix sums.

    Key insight:
      prefix[i] = arr[0] + arr[1] + … + arr[i-1]
      sum(arr[j..i-1]) = prefix[i] - prefix[j]
      We want prefix[i] - prefix[j] == k  ⟹  prefix[j] == prefix[i] - k

    Store seen prefix sums in a hash map.
    For each new prefix[i], look up (prefix[i] - k) in the map.
    O(n) time, O(n) space.
    """
    count   = 0
    prefix  = 0
    seen    = defaultdict(int)
    seen[0] = 1   # empty prefix sum = 0 seen once
    windows = []
    for i, val in enumerate(arr):
        prefix += val
        need    = prefix - k
        if need in seen:
            count += seen[need]
            # record ending positions
            windows.append(i)
        seen[prefix] += 1
    return count, windows

count, end_positions = subarray_sum_k(signal, TARGET_K)

print(f"Signal parameters:")
print(f"  Samples        : {N_SAMPLES}")
print(f"  Amplitude range: –{AMPLITUDE} to +{AMPLITUDE}")
print(f"  Target sum K   : {TARGET_K}")
print()
print(f"Signal: {signal}")
print()
print(f"Subarray sum = {TARGET_K}:")
print(f"  Count of subarrays: {count}")
print(f"  Ending at indices  : {end_positions[:10]}{'...' if len(end_positions)>10 else ''}")

# Verify by brute force
bf_count = sum(
    1 for i in range(N_SAMPLES) for j in range(i+1, N_SAMPLES+1)
    if sum(signal[i:j]) == TARGET_K
)
match_bf = (count == bf_count)
print(f"  Brute-force count  : {bf_count}")
print(f"  Hash-map matches   : {'✅ YES' if match_bf else '❌ NO'}")


## Step 2 — Rolling hash window (duplicate substring detection)

In [ ]:
def rolling_hash_windows(arr, width):
    """
    Slide a window of given width over the signal.
    Use a hash map to detect any repeated window pattern.

    Rolling hash update: remove leftmost element, add rightmost.
    Here we use Python's built-in hash on tuples for simplicity,
    but track the step-by-step process explicitly.

    Returns: {pattern: [start_indices]} for patterns seen more than once.
    """
    if width > len(arr): return {}
    window_map = defaultdict(list)   # pattern_hash → [start indices]
    for start in range(len(arr) - width + 1):
        window = tuple(arr[start:start + width])
        h      = hash(window)   # O(width) here; a true rolling hash would be O(1)
        window_map[h].append((start, window))
    # filter to duplicates only (same hash AND same values)
    duplicates = {}
    for h, entries in window_map.items():
        seen_windows = defaultdict(list)
        for start, w in entries:
            seen_windows[w].append(start)
        for w, starts in seen_windows.items():
            if len(starts) > 1:
                duplicates[w] = starts
    return duplicates

dupes = rolling_hash_windows(signal, WINDOW_WIDTH)

print(f"Rolling hash — width-{WINDOW_WIDTH} duplicate windows:")
print(f"  Total windows     : {N_SAMPLES - WINDOW_WIDTH + 1}")
print(f"  Unique repeated   : {len(dupes)}")
print()
if dupes:
    print(f"  {'Pattern':<40}  {'Starts'}")
    print(f"  {'─'*40}  {'─'*20}")
    for w, starts in list(dupes.items())[:5]:
        print(f"  {str(list(w)):<40}  {starts}")
    if len(dupes) > 5:
        print(f"  ... and {len(dupes)-5} more")
else:
    print("  No repeated windows found in this signal.")

most_repeated = max((len(v), k) for k, v in dupes.items())[1] if dupes else None
max_repeat_count = max((len(v) for v in dupes.values()), default=0)
print(f"\n  Most-repeated window appears: {max_repeat_count} times")


## Step 3 — Longest consecutive sequence (hash set, O(n))

In [ ]:
# Simulate a sample ID log: many IDs, some gaps
full_range  = list(range(1, N_CODES * 2))
sample_ids  = sorted(random.sample(full_range, N_CODES))
id_set      = set(sample_ids)

def longest_consecutive(nums):
    """
    Find the longest run of consecutive integers.

    Algorithm:
      1. Put all numbers in a hash set for O(1) lookup.
      2. For each number n, only START a chain if (n-1) is NOT in the set
         (so we start chains only at their beginning — O(n) total).
      3. Extend the chain by checking n+1, n+2, … until gap found.

    O(n) time, O(n) space.
    """
    num_set = set(nums)
    best_start, best_len = 0, 0
    for n in num_set:
        if (n - 1) not in num_set:   # n is the start of a chain
            cur, length = n, 1
            while (cur + 1) in num_set:
                cur    += 1
                length += 1
            if length > best_len:
                best_len, best_start = length, n
    return best_start, best_len

lcs_start, lcs_len = longest_consecutive(sample_ids)
lcs_sequence = list(range(lcs_start, lcs_start + lcs_len))

print(f"Sample ID log:")
print(f"  IDs in log         : {N_CODES}")
print(f"  ID range           : {min(sample_ids)} – {max(sample_ids)}")
print(f"  IDs present        : {sample_ids}")
print()
print(f"Longest consecutive sequence:")
print(f"  Start              : {lcs_start}")
print(f"  Length             : {lcs_len}")
print(f"  Sequence           : {lcs_sequence}")

# Gaps analysis
all_ids_in_range = set(range(min(sample_ids), max(sample_ids)+1))
gaps = sorted(all_ids_in_range - id_set)
print(f"\n  Gaps (missing IDs) : {len(gaps)}")
print(f"  Gap positions      : {gaps[:15]}{'...' if len(gaps)>15 else ''}")


## Step 4 — Word-pattern bijection check

In [ ]:
def word_pattern(pattern, sequence):
    """
    Check if sequence follows the same pattern as the template.
    'pattern' is a string like "aabba".
    'sequence' is a list of fault codes like ["E01","E01","E02","E02","E01"].

    Requires a BIJECTION: each pattern letter maps to exactly one code,
    AND each code maps to exactly one letter (no sharing).

    Two hash maps: letter→code and code→letter.
    O(n) time.
    """
    if len(pattern) != len(sequence):
        return False, "length mismatch"
    l2c = {}   # letter → code
    c2l = {}   # code → letter
    for letter, code in zip(pattern, sequence):
        if letter in l2c:
            if l2c[letter] != code:
                return False, f"letter '{letter}' mapped to '{l2c[letter]}' then '{code}'"
        else:
            l2c[letter] = code
        if code in c2l:
            if c2l[code] != letter:
                return False, f"code '{code}' mapped to '{c2l[code]}' then '{letter}'"
        else:
            c2l[code] = letter
    return True, l2c

# Generate a pattern and matching/non-matching sequences
PATTERN_LEN = random.randint(8, 14)
unique_letters = random.randint(2, 4)
letters = "abcde"[:unique_letters]
pattern = "".join(random.choices(letters, k=PATTERN_LEN))

codes_for_pattern = random.sample(CODE_POOL, unique_letters)
letter_to_code    = dict(zip(letters, codes_for_pattern))
matching_seq      = [letter_to_code[l] for l in pattern]

# Non-matching sequence (shuffle codes to break bijection)
shuffled_codes    = codes_for_pattern[:]
random.shuffle(shuffled_codes)
wrong_map         = dict(zip(letters, shuffled_codes))
nonmatch_seq      = [wrong_map[l] for l in pattern]

ok_true,  map_true  = word_pattern(pattern, matching_seq)
ok_false, map_false = word_pattern(pattern, nonmatch_seq)

print(f"Word-pattern bijection check:")
print(f"  Pattern           : {pattern}")
print()
print(f"  Matching sequence : {matching_seq}")
print(f"  Result            : {'✅ MATCH' if ok_true else '❌ NO MATCH'}")
if ok_true and isinstance(map_true, dict):
    for l, c in map_true.items():
        print(f"    '{l}' → {c}")
print()
print(f"  Non-matching seq  : {nonmatch_seq}")
print(f"  Result            : {'✅ MATCH' if ok_false else '❌ NO MATCH (expected)'}")
if not ok_false:
    print(f"  Reason            : {map_false}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " SIGNAL PATTERN ANALYSER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  SUBARRAY SUM = K" + " "*(W-18) + "║")
print(f"║  {'Signal length':<28}: {N_SAMPLES:<26}║")
print(f"║  {'Target K':<28}: {TARGET_K:<26}║")
print(f"║  {'Subarrays found':<28}: {count:<26}║")
print(f"║  {'Brute-force verified':<28}: {'YES ✅' if match_bf else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  ROLLING HASH WINDOW" + " "*(W-21) + "║")
print(f"║  {'Window width':<28}: {WINDOW_WIDTH:<26}║")
print(f"║  {'Repeated window patterns':<28}: {len(dupes):<26}║")
print(f"║  {'Max repeat count':<28}: {max_repeat_count:<26}║")
print("╠" + "═"*W + "╣")
print("║  LONGEST CONSECUTIVE" + " "*(W-21) + "║")
print(f"║  {'Sample IDs in log':<28}: {N_CODES:<26}║")
print(f"║  {'Longest consecutive run':<28}: {lcs_len} (start={lcs_start}){'':<14}║")
print(f"║  {'Gaps in ID range':<28}: {len(gaps):<26}║")
print("╠" + "═"*W + "╣")
print("║  WORD-PATTERN BIJECTION" + " "*(W-24) + "║")
print(f"║  {'Pattern length':<28}: {PATTERN_LEN:<26}║")
print(f"║  {'Matching seq result':<28}: {'MATCH ✅' if ok_true else 'NO MATCH ❌':<26}║")
print(f"║  {'Non-matching seq result':<28}: {'NO MATCH ✅ (expected)' if not ok_false else 'MATCH ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which hash table concept did you find most important, and why?**
Pick one technique (collision resolution, load factor, LRU cache, rolling hash…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
